In [1]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import GATConv
from datasets import CVFGATGeometricDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn.pool import global_mean_pool

In [2]:
device = "cuda"
batch_size = 8

In [3]:
dataset = CVFGATGeometricDataset(
    device, dataset="star_graph_n4", program="graph_coloring"
)  # list of Data objects
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [4]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1):
        super().__init__()
        # First GAT layer
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        # Second GAT layer
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        # x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        index = (
            torch.LongTensor([[i] * dataset[0].num_nodes for i in range(batch_size)])
            .to(device)
            .flatten()
        )
        return global_mean_pool(x, index).to(device)


# Model, optimizer, loss
model = GAT(
    in_channels=dataset.num_node_features,
    hidden_channels=8,
    out_channels=1,
    heads=8,
)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [ ]:
# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    total_loss = torch.FloatTensor([0.0]).to(device=device)
    for batch in loader:
        out = model(batch.x, batch.edge_index)
        loss = F.mse_loss(out.flatten(), batch.y)
        total_loss += loss
        loss.backward()
        optimizer.step()
    return total_loss.item()


# # Testing
# def test():
#     model.eval()
#     out = model(data.x, data.edge_index)
#     pred = out.argmax(dim=1)
#     accs = []
#     for mask in [data.train_mask, data.val_mask, data.test_mask]:
#         correct = (pred[mask] == data.y[mask]).sum()
#         acc = int(correct) / int(mask.sum())
#         accs.append(acc)
#     return accs


for epoch in range(1, 51):
    loss = train()
    # train_acc, val_acc, test_acc = test()
    if epoch % 5 == 0:
        # print(
        #     f"Epoch {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}"
        # )
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")

Epoch 005, Loss: 3.6680
Epoch 010, Loss: 2.2947
Epoch 015, Loss: 2.0534
Epoch 020, Loss: 1.8537
Epoch 025, Loss: 2.0128
Epoch 030, Loss: 2.1549
Epoch 035, Loss: 2.4305
Epoch 040, Loss: 1.8689
Epoch 045, Loss: 1.6610
Epoch 050, Loss: 1.6839
Epoch 055, Loss: 1.7081
Epoch 060, Loss: 1.6316
Epoch 065, Loss: 1.6729
Epoch 070, Loss: 1.2709
Epoch 075, Loss: 1.2448
Epoch 080, Loss: 1.1835
Epoch 085, Loss: 1.1234
Epoch 090, Loss: 1.1176
Epoch 095, Loss: 1.2274
Epoch 100, Loss: 1.3664
